# Single-axis SMA scan

Fifty runs of the same minimal LEO mission, one per semi-major-axis value across `np.linspace(7000, 8000, 50)`, fanned out in parallel and overlaid on a single altitude-vs-time plot.

This is the smallest end-to-end exercise of the v0.1 surface: one [`sweep()`][gmat_sweep.sweep] call returns a `(run_id, time)`-MultiIndexed DataFrame; the plot below loops over `run_id` and overlays each per-run trajectory.

**Prerequisites.** A local GMAT install (R2026a is the primary development target; see [Supported versions](../supported-versions.md)) and `pip install gmat-sweep[examples]` for the matplotlib dependency.

## Set up the run

Resolve the GMAT install once and confirm the script that ships next to this notebook is where we expect it. The script lives in the same directory so the path is machine-independent.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from gmat_run import locate_gmat

from gmat_sweep import sweep

install = locate_gmat()
script_path = Path("leo_keplerian.script").resolve()

print(f"GMAT version: {install.version}")
print(f"Script:       {script_path.name}")
print(f"Exists:       {script_path.exists()}")

## Define the grid

Fifty semi-major-axis values, evenly spaced from 7000 km (~622 km altitude) to 8000 km (~1622 km altitude). The grid is a plain dict from dotted-path field name to an iterable of values; `sweep()` materialises the cartesian product internally.

In [ ]:
sma_values = np.linspace(7000.0, 8000.0, 50)
sma_values

## Run the sweep

One [`sweep()`][gmat_sweep.sweep] call dispatches all 50 runs through the default `LocalJoblibPool`, drains the per-run outcomes in completion order, and returns the aggregated `(run_id, time)`-MultiIndexed DataFrame.

Each row carries the corresponding `Sat.SMA` override applied to that run, the `__status` column (`ok` / `failed` / `skipped`), and one column per channel the script's `ReportFile` declared. With `out=None` the per-run Parquet files land in a temporary directory whose lifetime is tied to the returned DataFrame.

In [ ]:
df = sweep(script_path, grid={"Sat.SMA": sma_values}, workers=-1)
df.head()

Confirm every run finished cleanly. Failed and skipped runs would surface as one NaN-filled row each, with `__status` set accordingly.

In [ ]:
df["__status"].value_counts()

## Derive altitude and overlay

The script's ReportFile records inertial position only (`Sat.X`, `Sat.Y`, `Sat.Z`) — altitude is a derived quantity. For a spherical Earth it's $|\mathbf{r}| - R_\oplus$. We compute it on the whole frame in one vectorised pass; the `(run_id, time)` MultiIndex carries through unchanged.

In [ ]:
EARTH_RADIUS_KM = 6378.137  # WGS-84 equatorial radius

position = df[["Sat.X", "Sat.Y", "Sat.Z"]]
df["Altitude_km"] = (position**2).sum(axis=1) ** 0.5 - EARTH_RADIUS_KM

Overlay every run on a single axis. Grouping by `run_id` lets us iterate one trajectory at a time; we colour each line by its commanded `Sat.SMA` so the visual sweep matches the parameter sweep.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
cmap = plt.get_cmap("viridis")

for run_id, group in df.groupby(level="run_id"):
    sma = sma_values[run_id]
    color = cmap((sma - sma_values.min()) / (sma_values.max() - sma_values.min()))
    times = group.index.get_level_values("time")
    ax.plot(times, group["Altitude_km"], color=color, alpha=0.7, linewidth=1.0)

norm = plt.Normalize(vmin=sma_values.min(), vmax=sma_values.max())
fig.colorbar(
    plt.cm.ScalarMappable(norm=norm, cmap=cmap),
    ax=ax,
    label="Sat.SMA (km)",
)
ax.set_xlabel("UTC")
ax.set_ylabel("Altitude (km)")
ax.set_title("50 SMA-scan runs overlaid (1 hour propagation each)")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## Where to next

- **Two-axis grids.** [Notebook 02](02_epoch_arrival_grid.ipynb) adds a second sweep axis (launch epoch × time-of-flight) and contours a per-run scalar across the grid.
- **Inspect a partial sweep after a kill.** [Notebook 03](03_killed_sweep_recovery.ipynb) demonstrates that the JSON Lines manifest survives a mid-sweep `Ctrl-C` and can be loaded back from disk.
- **Keep the per-run Parquet files.** Pass `out=Path("./sweep-out")` to [`sweep()`][gmat_sweep.sweep] to anchor the per-run artefacts under an explicit directory instead of a temp dir.
- **CLI alternative.** The [`gmat-sweep run`](../parameter-spec.md) shell command wraps the same call for non-Python pipelines.